# Questão 8 - Sistema de recomendação

O objetivo desta etapa é implementar uma vitrine de recomendação ("Quem comprou isso, também levou...") para aumentar o ticket médio, começando pelo item GPS Garmin Vortex Maré Drift (ID 27)
Este motor utiliza a similaridade de comportamento de compra para sugerir itens complementares ou de interesse comum.

### 1. Carregamento dos dados

In [1]:
#importando bibliotecas
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Carregando as bases brutas fornecidas
df_vendas = pd.read_csv('datasets/vendas_2023_2024.csv')
df_produtos = pd.read_csv('datasets/produtos_raw.csv')

In [3]:
# Visualizando as primeiras linhas para validação
df_vendas.head()

,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


In [4]:
df_vendas.drop(columns = "id")

,id_client,id_product,qtd,total,sale_date
0,42,105,11,3405.00,2023-09-10
1,3,136,9,16873.90,15-09-2024
2,25,139,7,9475.30,2024-08-13
3,20,23,5,55893.00,2023-02-03
4,8,57,4,451403.90,2024-02-12
...,...,...,...,...,...
9890,30,139,6,8549.00,11-03-2023
9891,9,111,7,28497.15,17-09-2023
9892,38,123,2,5276.30,20-06-2023
9893,33,97,6,771409.50,23-10-2024


In [5]:
df_vendas.set_index("id")

,id_client,id_product,qtd,total,sale_date
id,,,,,
0,42,105,11,3405.00,2023-09-10
1,3,136,9,16873.90,15-09-2024
2,25,139,7,9475.30,2024-08-13
4,20,23,5,55893.00,2023-02-03
5,8,57,4,451403.90,2024-02-12
...,...,...,...,...,...
9995,30,139,6,8549.00,11-03-2023
9996,9,111,7,28497.15,17-09-2023
9997,38,123,2,5276.30,20-06-2023


### 2. Construção da Matriz de Interação Usuário × Produto
Conforme a regra do negócio, criamos uma matriz onde as linhas são os clientes e as colunas são os produtos. O valor é 1 se houve compra (independente da quantidade) e 0 caso contrário.

In [6]:
# Criando a tabela de cruzamento (Crosstab)
matrix = pd.crosstab(df_vendas['id_client'], df_vendas['id_product'])

# Convertendo para presença (1) ou ausência (0)
matrix = (matrix > 0).astype(int)

print(f"Dimensões da Matriz: {matrix.shape}")

Dimensões da Matriz: (49, 150)


### 3. Cálculo de Similaridade de Cosseno entre Produtos 
Calculamos a similaridade técnica entre os vetores de cada produto para entender quais itens costumam "andar juntos" nos carrinhos de compra.

In [7]:
# Transpomos a matriz para calcular a similaridade produto x produto
# O cálculo de cosseno mede o ângulo entre os vetores de compradores
product_sim = cosine_similarity(matrix.T)

# Transformamos em um DataFrame para facilitar a consulta por ID
sim_df = pd.DataFrame(product_sim, index=matrix.columns, columns=matrix.columns)

In [8]:
sim_df

id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_product,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.775058,0.737865,0.810191,0.748331,0.769484,0.775058,0.727825,0.711512,0.698771,...,0.795133,0.753819,0.775058,0.769484,0.721605,0.775058,0.872082,0.721605,0.727825,0.775058
2,0.775058,1.000000,0.704295,0.757865,0.714286,0.712931,0.771429,0.750290,0.788811,0.687256,...,0.795192,0.778078,0.771429,0.767772,0.685714,0.742857,0.712931,0.628571,0.750290,0.742857
3,0.737865,0.704295,1.000000,0.800641,0.704295,0.865181,0.732467,0.712396,0.777778,0.707107,...,0.675923,0.739795,0.732467,0.757033,0.704295,0.788811,0.729996,0.732467,0.739795,0.704295
4,0.810191,0.757865,0.800641,1.000000,0.757865,0.753310,0.757865,0.789747,0.773953,0.735980,...,0.831239,0.789747,0.730798,0.779287,0.730798,0.730798,0.805263,0.757865,0.763422,0.757865
5,0.748331,0.714286,0.704295,0.757865,1.000000,0.795192,0.742857,0.694713,0.760639,0.627495,...,0.795192,0.722501,0.714286,0.685511,0.685714,0.714286,0.767772,0.657143,0.722501,0.714286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,0.775058,0.742857,0.788811,0.730798,0.714286,0.767772,0.714286,0.722501,0.760639,0.627495,...,0.712931,0.722501,0.714286,0.740351,0.714286,1.000000,0.712931,0.771429,0.722501,0.657143
147,0.872082,0.712931,0.729996,0.805263,0.767772,0.763158,0.767772,0.720064,0.702959,0.688247,...,0.789474,0.720064,0.795192,0.763158,0.740351,0.712931,1.000000,0.685511,0.746733,0.795192
148,0.721605,0.628571,0.732467,0.757865,0.657143,0.767772,0.685714,0.750290,0.788811,0.687256,...,0.685511,0.805867,0.714286,0.767772,0.771429,0.771429,0.685511,1.000000,0.750290,0.714286


### 4. Geração de Ranking para o GPS Garmin (ID 27)

In [9]:
# Definindo o item de referência
ref_id = 27 

if ref_id in sim_df.columns:
    # Ordena os produtos por maior similaridade
    ranking = sim_df[ref_id].sort_values(ascending=False)
    
    # Remove o próprio item da lista de recomendações
    ranking = ranking.drop(labels=[ref_id])
    
    # Seleciona os top 5 produtos similares
    top_5 = ranking.head(5)
    
    print(f"Top 5 produtos recomendados para quem comprou o item {ref_id}:")
    print(top_5)
else:
    print(f"Erro: O produto ID {ref_id} não possui registros de vendas para análise")

Top 5 produtos recomendados para quem comprou o item 27:
id_product
94     0.869626
11     0.868037
35     0.853913
115    0.850000
1      0.850000
Name: 27, dtype: float64


### 5. Validação de Nomes 

In [11]:
# Cruzando o Top 5 com a base de nomes de produtos
top_5_names = df_produtos[df_produtos['code'].isin(top_5.index)][['code', 'name']]
print("Sugestões para a vitrine do site:")
print(top_5_names)


Sugestões para a vitrine do site:
     code                                        name
0       1                 Transponder AIS Maré Magnum
10     11         GPS Furuno Swift Leviathan Poseidon
34     35                          Radar Furuno Swift
97     94            Motor de Popa Volvo Magnum 276HP
118   115  Cabo de Nylon Delta Force Magnum Leviathan


### 6. Considerações Finais

***Por que o Motor de Popa (ID 94) ficou em primeiro?*** O resultado demonstra o que discutimos sobre as limitações do modelo. O produto 94 é um item de alta frequência de vendas na base de dados (comprado por muitos clientes como o ID 41, 24, 13 e 15). Como muitas pessoas compram motores, a probabilidade matemática de um comprador de GPS também ter comprado um motor é muito alta, gerando uma similaridade de cosseno elevada.

***Interpretação da Similaridade de Cosseno:*** Neste contexto, os scores acima de 0.80 indicam uma fortíssima sobreposição de público. Isso significa que mais de 80% do "perfil de compra" dos clientes que levaram o GPS (ID 27) é idêntico ao perfil de quem levou o Motor Volvo (ID 94) ou o GPS Furuno (ID 11).

***Insights para a Marina:*** Embora o motor (ID 94) seja o mais similar matematicamente, a recomendação do Cabo de Nylon (ID 115) ou do Radar Furuno (ID 35) pode ser mais estratégica para o e-commerce, pois são itens acessórios que complementam a instalação do GPS ou a navegação, enquanto o motor é uma compra de categoria principal
.
